<a href="https://colab.research.google.com/github/amitmishra-cse/Relative-Deprivation-in-Hindu-and-Muslims/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import pandas as pd
import numpy as np
from google.colab import files

# Ensure we have our dataframe loaded
try:
    if 'df' not in globals():
        print("Please upload your .xlsx file first:")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename.endswith('.xlsx'):
                df = pd.read_excel(io.BytesIO(uploaded[filename]))
                print(f"Successfully loaded '{filename}'!")
                break
except Exception as e:
    print(f"Error loading dataset: {e}")

if 'df' in globals():
    print("===============================================")
    print("             TECHNICAL VALIDATION REPORT       ")
    print("===============================================\n")

    # 1. Structural Integrity & Shape Check
    print("--- 1. STRUCTURAL INTEGRITY ---")
    n_rows, n_cols = df.shape
    print(f"[PASS] Dataset shape: {n_rows} rows, {n_cols} columns.")
    print(f"[INFO] Unique primary identifiers (id): {df['id'].nunique() if 'id' in df.columns else 'N/A'}")

    # 2. Duplicate Check
    print("\n--- 2. DUPLICITY VERIFICATION ---")
    duplicate_count = df.duplicated().sum()
    if duplicate_count == 0:
        print("[PASS] No completely duplicate rows detected.")
    else:
        print(f"[WARN] {duplicate_count} duplicate rows found in the dataset!")

    # 3. Key Demographics Consistency Check
    print("\n--- 3. CATEGORICAL FIELD CONSTRAINTS ---")
    demo_fields = ['Gender', 'Religion', 'Residence']
    for field in demo_fields:
        if field in df.columns:
            unique_vals = df[field].dropna().unique()
            val_counts = df[field].value_counts(dropna=False).to_dict()
            print(f"[PASS] '{field}' contains {len(unique_vals)} unique values: {val_counts}")
        else:
            print(f"[FAIL] Key demographic field '{field}' is missing!")

    # 4. Out-of-Range Bounds Check (Continuous Variables)
    print("\n--- 4. BOUNDS & RANGE VALIDITY ---")
    # Age validation
    if 'Age' in df.columns:
        min_age, max_age = df['Age'].min(), df['Age'].max()
        if min_age < 15 or max_age > 100:
            print(f"[WARN] Age range contains potential anomalies: Min={min_age}, Max={max_age}")
        else:
            print(f"[PASS] Age range appears normal: Min={min_age}, Max={max_age}")

    # Psychometric Scale Validation (Checking if scores fall within expected 1-5 range)
    psych_scales = ['cognitivefrd', 'affectivefrd', 'Cognitiveegoisticrd', 'Affectiveegoisticrd']
    for scale in psych_scales:
        if scale in df.columns:
            col_data = df[scale].dropna()
            out_of_bounds = col_data[(col_data < 1.0) | (col_data > 5.0)]
            if len(out_of_bounds) > 0:
                print(f"[FAIL] Scale '{scale}' has {len(out_of_bounds)} values out of standard bounds (1.0 - 5.0)!")
            else:
                print(f"[PASS] Scale '{scale}' values strictly within boundaries (1.0 - 5.0). Min={col_data.min():.2f}, Max={col_data.max():.2f}")

    # 5. Missingness Analysis
    print("\n--- 5. MISSINGNESS ANALYSIS ---")
    col_missing = df.isnull().sum()
    cols_with_missing = col_missing[col_missing > 0]
    if len(cols_with_missing) > 0:
        print(f"[INFO] {len(cols_with_missing)} columns contain missing values.")
        print("Top 5 columns with highest missing counts:")
        print(cols_with_missing.sort_values(ascending=False).head(5))
    else:
        print("[PASS] Zero missing values detected across the entire dataset!")

    # 6. Data Type Homogeneity Check
    print("\n--- 6. COLUMN DATA TYPE CONFORMANCE ---")
    mixed_types = []
    for col in df.columns:
        # Infers if a column has multiple types of entries
        inferred_type = pd.api.types.infer_dtype(df[col], skipna=True)
        if inferred_type == 'mixed':
            mixed_types.append(col)

    if len(mixed_types) > 0:
        print(f"[WARN] {len(mixed_types)} columns have mixed python types (could lead to computation bugs):")
        print(mixed_types[:10])
    else:
        print("[PASS] All columns conform to homogeneous data types.")

else:
    print("No active dataset loaded in memory.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

demo_df = df[['Gender', 'Religion', 'Residence']].melt(var_name='Demographic', value_name='Category').dropna()
sns.countplot(data=demo_df, x='Demographic', hue='Category', palette="Set2")

plt.title("Demographic & Geographical Balance Check", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Demographic Dimension", fontsize=12)
plt.ylabel("Respondent Count", fontsize=12)
plt.legend(title="Subgroup Category", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
print(f"Original shape: {df.shape}")
# Filter out any records where Age is less than 12 (which removes the outlier of 2)
df = df[df['Age'] >= 12]
print(f"New shape: {df.shape}")
print(f"Updated Age range: Min={df['Age'].min()}, Max={df['Age'].max()}")

In [ ]:
import numpy as np
import pandas as pd

# 1. Identify and impute missing values in numeric columns with the median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().any():
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)

# 2. Identify and impute missing values in categorical/object columns with the mode
categorical_cols = df.select_dtypes(exclude=[np.number]).columns
for col in categorical_cols:
    if df[col].isnull().any():
        mode_series = df[col].mode()
        if not mode_series.empty:
            df[col] = df[col].fillna(mode_series[0])

# Verify that all missing values have been successfully addressed
remaining_missing = df.isnull().sum().sum()
print(f"Imputation completed successfully!")
print(f"Remaining missing values in the dataset: {remaining_missing}")

In [ ]:
# --- Explain the validation results in markdown or python console ---
print("Validation Summary:")
print("- Data dimensions: 300 rows, 149 columns")
print("- No duplicates found")
print("- Clean demographic splits (Gender: 165M/135F; Religion: 154M/146H)")
print("- Age range cleaned (18 to 68 years)")
print("- Psychometric scale ratings bounded strictly within 1.0 - 5.0")
print("- Missingness resolved: 0 remaining missing values!")

In [ ]:
# Export the cleaned DataFrame to an Excel file
export_filename = 'cleaned_dataset.xlsx'
df.to_excel(export_filename, index=False)

# Trigger browser download of the cleaned dataset
from google.colab import files
files.download(export_filename)